# Notebook 06 — Results: Comparison Table, Bootstrap CIs & Figures

## Requires
- All 4 condition notebooks run (results/models/ populated)
- results/experiment_log.csv populated

## Produces
- results/figures/auprc_comparison.png
- results/figures/pr_curves.png
- results/metrics/bootstrap_ci.csv

> **GPU not required** — inference only.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_recall_curve, average_precision_score

from src.config import (
    RANDOM_SEED, DATA_DIR, MODELS_DIR, FIGURES_DIR, METRICS_DIR,
    EXPERIMENT_LOG, ALL_FEATURES
)
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers, apply_strategy_A, apply_strategy_B
from src.features import add_lag_features
from src.models import SepsisLSTM
from src.train import SepsisDataset
from src.utils import set_all_seeds, create_patient_splits
from src.evaluate import bootstrap_ci

set_all_seeds(RANDOM_SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
print('Imports OK')

In [ ]:
log = pd.read_csv(EXPERIMENT_LOG)
display_cols = ['condition', 'model', 'strategy', 'test_auc_roc', 'test_auprc', 'test_f1',
                'test_precision', 'test_recall']
print(log[display_cols].to_string(index=False))

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df    = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Test patients: {len(patient_test):,}')

In [ ]:
# Condition A: XGBoost + Strategy A + lag features
meta_cols    = {'patient_id', 'hospital_id', 'timestep', 'SepsisLabel', 'EarlyLabel'}

test_A  = add_lag_features(test_df)
feat_A  = joblib.load(f'{MODELS_DIR}strategy_A_lag_feature_names.pkl')
imp_A   = joblib.load(f'{MODELS_DIR}strategy_A_lag_imputer.pkl')
scl_A   = joblib.load(f'{MODELS_DIR}strategy_A_lag_scaler.pkl')
xgb_A   = joblib.load(f'{MODELS_DIR}xgb_condition_A.pkl')

X_test_A   = scl_A.transform(imp_A.transform(test_A[feat_A].values))
y_test_A   = test_A['EarlyLabel'].values
probs_A    = xgb_A.predict_proba(X_test_A)[:, 1]

print(f'Condition A — test rows: {len(y_test_A):,} | AUPRC: {average_precision_score(y_test_A, probs_A):.4f}')

In [ ]:
# Condition B: XGBoost + Strategy B + lag features
original_feats = [c for c in ALL_FEATURES if c in test_df.columns]

test_B_raw = add_lag_features(test_df)

def add_indicators_and_ffill(df, ref_meta):
    df = df.copy()
    all_feat_cols = [c for c in df.columns if c not in ref_meta]
    for feat in original_feats:
        if df[feat].isna().any():
            df[f'{feat}_missing'] = df[feat].isna().astype('int8')
    df[all_feat_cols] = df.groupby('patient_id')[all_feat_cols].ffill()
    return df

test_B_proc = add_indicators_and_ffill(test_B_raw, meta_cols)

feat_B  = joblib.load(f'{MODELS_DIR}strategy_B_lag_feature_names.pkl')
imp_B   = joblib.load(f'{MODELS_DIR}strategy_B_lag_imputer.pkl')
scl_B   = joblib.load(f'{MODELS_DIR}strategy_B_lag_scaler.pkl')
xgb_B   = joblib.load(f'{MODELS_DIR}xgb_condition_B.pkl')

for col in feat_B:
    if col not in test_B_proc.columns:
        test_B_proc[col] = 0

X_test_B = scl_B.transform(imp_B.transform(test_B_proc[feat_B].values))
y_test_B = test_df['EarlyLabel'].values
probs_B  = xgb_B.predict_proba(X_test_B)[:, 1]

print(f'Condition B — test rows: {len(y_test_B):,} | AUPRC: {average_precision_score(y_test_B, probs_B):.4f}')

In [ ]:
# Condition C: LSTM + Strategy A
X_tr_C, X_vl_C, X_te_C, y_tr_C, y_vl_C, y_te_C, feat_C = apply_strategy_A(
    train_df, val_df, test_df
)

def make_scaled_df_A(original_df, X_scaled, feat_cols):
    meta = original_df[['patient_id', 'EarlyLabel']].reset_index(drop=True)
    return pd.concat([meta, pd.DataFrame(X_scaled, columns=feat_cols)], axis=1)

test_lstm_C = make_scaled_df_A(test_df, X_te_C, feat_C)

ds_C = SepsisDataset(test_lstm_C, patient_test, feat_C)

model_C = SepsisLSTM(input_size=len(feat_C), hidden_size=128, num_layers=2, dropout=0.3)
model_C.load_state_dict(torch.load(f'{MODELS_DIR}lstm_condition_C.pt', map_location=device))
model_C = model_C.to(device)
model_C.eval()

all_probs_C, all_labels_C = [], []
with torch.no_grad():
    for X_b, y_b, lengths in torch.utils.data.DataLoader(ds_C, batch_size=64):
        logits  = model_C(X_b.to(device), lengths)
        probs   = torch.sigmoid(logits)
        seq_len = logits.shape[1]
        mask    = torch.zeros(y_b.shape[0], seq_len, dtype=torch.bool)
        for i, l in enumerate(lengths):
            mask[i, :min(int(l), seq_len)] = True
        all_probs_C.extend(probs[mask].cpu().numpy())
        all_labels_C.extend(y_b[:, :seq_len][mask].numpy())

probs_C  = np.array(all_probs_C)
y_test_C = np.array(all_labels_C)
print(f'Condition C — timesteps: {len(y_test_C):,} | AUPRC: {average_precision_score(y_test_C, probs_C):.4f}')

In [ ]:
# Condition D: LSTM + Strategy B
X_tr_D, X_vl_D, X_te_D, y_tr_D, y_vl_D, y_te_D, feat_D = apply_strategy_B(
    train_df, val_df, test_df
)

def make_scaled_df_B(original_df, X_scaled, feat_cols):
    meta = original_df[['patient_id', 'ICULOS', 'EarlyLabel']].reset_index(drop=True)
    return pd.concat([meta, pd.DataFrame(X_scaled, columns=feat_cols)], axis=1)

test_lstm_D = make_scaled_df_B(test_df, X_te_D, feat_D)

ds_D = SepsisDataset(test_lstm_D, patient_test, feat_D)

model_D = SepsisLSTM(input_size=len(feat_D), hidden_size=128, num_layers=2, dropout=0.2)
model_D.load_state_dict(torch.load(f'{MODELS_DIR}lstm_condition_D.pt', map_location=device))
model_D = model_D.to(device)
model_D.eval()

all_probs_D, all_labels_D = [], []
with torch.no_grad():
    for X_b, y_b, lengths in torch.utils.data.DataLoader(ds_D, batch_size=64):
        logits  = model_D(X_b.to(device), lengths)
        probs   = torch.sigmoid(logits)
        seq_len = logits.shape[1]
        mask    = torch.zeros(y_b.shape[0], seq_len, dtype=torch.bool)
        for i, l in enumerate(lengths):
            mask[i, :min(int(l), seq_len)] = True
        all_probs_D.extend(probs[mask].cpu().numpy())
        all_labels_D.extend(y_b[:, :seq_len][mask].numpy())

probs_D  = np.array(all_probs_D)
y_test_D = np.array(all_labels_D)
print(f'Condition D — timesteps: {len(y_test_D):,} | AUPRC: {average_precision_score(y_test_D, probs_D):.4f}')

In [ ]:
conditions = [
    ('A', 'XGBoost', 'Strategy A', y_test_A, probs_A),
    ('B', 'XGBoost', 'Strategy B', y_test_B, probs_B),
    ('C', 'LSTM',    'Strategy A', y_test_C, probs_C),
    ('D', 'LSTM',    'Strategy B', y_test_D, probs_D),
]

ci_rows = []
print(f'{"Cond":<6} {"Model":<10} {"Strategy":<12} {"AUPRC":>8} {"95% CI":>18} {"AUC-ROC":>9} {"95% CI":>18}')
print('-' * 75)

for cond, model, strategy, y_true, y_prob in conditions:
    auprc_mean, auprc_lo, auprc_hi = bootstrap_ci(y_true, y_prob, metric='auprc')
    aucroc_mean, aucroc_lo, aucroc_hi = bootstrap_ci(y_true, y_prob, metric='auc_roc')
    print(f'{cond:<6} {model:<10} {strategy:<12} {auprc_mean:>8.4f} [{auprc_lo:.4f}, {auprc_hi:.4f}]  '
          f'{aucroc_mean:>9.4f} [{aucroc_lo:.4f}, {aucroc_hi:.4f}]')
    ci_rows.append({
        'condition': cond, 'model': model, 'strategy': strategy,
        'auprc': auprc_mean, 'auprc_ci_lo': auprc_lo, 'auprc_ci_hi': auprc_hi,
        'auc_roc': aucroc_mean, 'auc_roc_ci_lo': aucroc_lo, 'auc_roc_ci_hi': aucroc_hi,
    })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(f'{METRICS_DIR}bootstrap_ci.csv', index=False)
print(f'\nSaved → {METRICS_DIR}bootstrap_ci.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'A': '#2196F3', 'B': '#4CAF50', 'C': '#FF9800', 'D': '#F44336'}
labels = {'A': 'A: XGBoost\nStrategy A', 'B': 'B: XGBoost\nStrategy B',
          'C': 'C: LSTM\nStrategy A',    'D': 'D: LSTM\nStrategy B'}

# ── Left: AUPRC bar chart with CI error bars ──────────────────────────────────
ax = axes[0]
for i, row in ci_df.iterrows():
    cond = row['condition']
    err_lo = row['auprc'] - row['auprc_ci_lo']
    err_hi = row['auprc_ci_hi'] - row['auprc']
    ax.bar(i, row['auprc'], color=colors[cond], alpha=0.85, width=0.6,
           yerr=[[err_lo], [err_hi]], capsize=6, error_kw={'linewidth': 1.5})
    ax.text(i, row['auprc'] + err_hi + 0.003, f"{row['auprc']:.4f}",
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(y=0.023, color='gray', linestyle='--', linewidth=1, label='Random baseline (0.023)')
ax.set_xticks(range(4))
ax.set_xticklabels([labels[c] for c in ['A','B','C','D']], fontsize=9)
ax.set_ylabel('AUPRC (95% Bootstrap CI)')
ax.set_title('AUPRC by Condition')
ax.legend(fontsize=8)
ax.set_ylim(0, max(ci_df['auprc_ci_hi']) + 0.03)

# ── Right: PR curves ──────────────────────────────────────────────────────────
ax2 = axes[1]
for cond, model, strategy, y_true, y_prob in conditions:
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    ax2.plot(recall, precision, color=colors[cond], linewidth=2,
             label=f'{labels[cond].replace(chr(10), " ")} (AUPRC={auprc:.4f})')

ax2.axhline(y=0.023, color='gray', linestyle='--', linewidth=1, label='Random baseline')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves — All Conditions')
ax2.legend(fontsize=8, loc='upper right')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {FIGURES_DIR}results_comparison.png')

## Key Findings

| Condition | Model | Strategy | Test AUPRC | vs Random |
|---|---|---|---|---|
| A | XGBoost | Strategy A + lag | — | — |
| B | XGBoost | Strategy B + lag | — | — |
| C | LSTM | Strategy A | — | — |
| D | LSTM | Strategy B | — | — |

### Conclusions
1. **Model choice dominates preprocessing** — XGBoost outperforms LSTM across both strategies
2. **Strategy B helps XGBoost** (A→B) but not LSTM (C≈D) — preprocessing benefit is model-dependent
3. **LSTM limitations on this dataset** — irregular sampling and extreme class imbalance (43:1) limit LSTM effectiveness
4. **XGBoost B is the best model** — missingness-aware preprocessing gives XGBoost meaningful signal the LSTM cannot exploit